# Similarity Search

Embed a query reaction SMARTS and find the most similar reactions in the precomputed embedding index using cosine similarity.

> Requires `uv sync --extra all`, plus:
> - `data/embeddings/medium/{embeddings.npy,smarts.txt}` — download from [Zenodo](https://doi.org/10.5281/zenodo.22645328), or regenerate (see README's Data pipeline and Training sections).
> - `data/raw/retrorules-v3.0-{metanetx,rhea}.csv` — download from [retrorules.org](https://retrorules.org/) (for EC-class labels).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rxn_smarts_embeddings.predict import load_embedder

## Load the embedding index

In [ ]:
EMB_PATH    = "data/embeddings/medium/embeddings.npy"
SMARTS_PATH = "data/embeddings/medium/smarts.txt"
RAW_FILES   = [
    "data/raw/retrorules-v3.0-metanetx.csv",
    "data/raw/retrorules-v3.0-rhea.csv",
]

index_embs  = np.load(EMB_PATH).astype(np.float32)           # (N, d_model)
index_smarts = Path(SMARTS_PATH).read_text().splitlines()

# pre-normalise for fast cosine similarity via dot product
norms = np.linalg.norm(index_embs, axis=1, keepdims=True)
index_normed = index_embs / np.clip(norms, 1e-9, None)

print(f"Index: {index_normed.shape[0]:,} reactions  dim={index_normed.shape[1]}")

In [ ]:
# load EC labels for display
raw = pd.concat([pd.read_csv(f) for f in RAW_FILES], ignore_index=True)
raw = raw[["TEMPLATE", "ECS"]].dropna(subset=["TEMPLATE"]).drop_duplicates("TEMPLATE")
smarts_to_ec = raw.set_index("TEMPLATE")["ECS"].to_dict()

## Search function

In [ ]:
embedder = load_embedder(pooling="cls")

def search(query_smarts: str, top_k: int = 10, exclude_self: bool = True):
    """Return the top-k most similar reactions to query_smarts."""
    q_emb = embedder.embed([query_smarts])[0].astype(np.float32)   # (d,)
    q_emb /= np.linalg.norm(q_emb) + 1e-9

    sims = index_normed @ q_emb                                     # (N,)

    if exclude_self and query_smarts in index_smarts:
        self_idx = index_smarts.index(query_smarts)
        sims[self_idx] = -1.0

    top_idx = np.argsort(sims)[::-1][:top_k]

    return pd.DataFrame({
        "rank":       range(1, top_k + 1),
        "similarity": sims[top_idx].round(4),
        "smarts":     [index_smarts[i] for i in top_idx],
        "ec":         [smarts_to_ec.get(index_smarts[i], "") for i in top_idx],
    })

## Example query

Search for reactions similar to an imine reduction: `C=N → C-N`.

In [ ]:
query = "[C;H1:1]=[N;H0:2]>>[C;H1:1]-[N;H0:2]"

results = search(query, top_k=10)
pd.set_option("display.max_colwidth", 80)
results

## Visualise similarity distribution

In [ ]:
q_emb = embedder.embed([query])[0].astype(np.float32)
q_emb /= np.linalg.norm(q_emb) + 1e-9
all_sims = index_normed @ q_emb

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(all_sims, bins=100, color="steelblue", edgecolor="none")
for sim in results["similarity"].values[:5]:
    ax.axvline(sim, color="crimson", lw=1, alpha=0.7)
ax.set_xlabel("cosine similarity to query")
ax.set_ylabel("count")
ax.set_title(f"Similarity distribution — query: {query[:50]}…")
plt.tight_layout()
plt.show()

## Try your own query

In [ ]:
my_query = "[C:1]-[O:2]>>[C:1]=[O:2]"   # ← change this

search(my_query, top_k=8)

## Compare two reactions in embedding space

In [ ]:
rxn_a = "[C;H1:1]=[N;H0:2]>>[C;H1:1]-[N;H0:2]"    # imine reduction
rxn_b = "[C;H0:1]=[O:2]>>[C;H1:1]-[O:2]"           # ketone reduction
rxn_c = "c1ccccc1>>c1cccnc1"                         # benzene → pyridine

embs_cmp = embedder.embed([rxn_a, rxn_b, rxn_c])
embs_cmp /= np.linalg.norm(embs_cmp, axis=1, keepdims=True)
sim_mat = embs_cmp @ embs_cmp.T

names = ["imine reduction", "ketone reduction", "arene → heteroarene"]
fig, ax = plt.subplots(figsize=(4, 3))
im = ax.imshow(sim_mat, vmin=0, vmax=1, cmap="YlOrRd")
ax.set_xticks(range(3), names, rotation=25, ha="right", fontsize=8)
ax.set_yticks(range(3), names, fontsize=8)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{sim_mat[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()